# 00 - Esplorazione del dataset R&D (LHC Olympics 2020)

Obiettivi di questo notebook:

1. verificare che i dati siano stati letti correttamente;
2. guardare la distribuzione di $m_{JJ}$ e localizzare il segnale;
3. **misurare la correlazione tra le variabili ausiliarie $Y$ e $m_{JJ}$ sul solo fondo.**

Il punto 3 e' il piu' importante: e' il controllo che decide se il metodo CWoLa puo' funzionare.
Se le $Y$ sono correlate con $m_{JJ}$, il classificatore impara la massa invece della
sottostruttura e scolpisce un bump artificiale nel fondo.

Prerequisito: `python scripts/prepare_data.py` gia' eseguito.

In [ ]:
import sys
sys.path.insert(0, "..")
%load_ext autoreload
%autoreload 2

import numpy as np
import matplotlib.pyplot as plt

from src.config import load_config, processed_path
from src.io import load_processed, LABEL_COL
from src.features import (AUX_FEATURES, AUX_FEATURES_EXTENDED, RESONANT_FEATURE,
                          correlation_with_mjj, sanity_report)

cfg = load_config()
FIGDIR = cfg["paths"]["figures"]
FIGDIR.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({"figure.dpi": 110, "font.size": 11,
                     "axes.grid": True, "grid.alpha": 0.3})

In [ ]:
df = load_processed(processed_path(cfg, "processed_main"))
print(df.shape)
df.head()

## 1. Controlli di sanita'

Se uno di questi non torna, il problema e' a monte (colonne scambiate, chiave sbagliata)
e non ha senso proseguire. In particolare `n_signal` deve fare esattamente 100 000.

In [ ]:
for k, v in sanity_report(df).items():
    print(f"{k:.<34} {v}")

## 2. La variabile risonante $m_{JJ}$

Il fondo QCD e' una legge di potenza liscia e ripida; il segnale e' una gaussiana
allargata attorno a 3.5 TeV. Le linee verticali sono le finestre SR/SB del config.

In [ ]:
phys = cfg["physics"]
bins = np.arange(phys["mjj_range"][0], phys["mjj_range"][1] + 1,
                 phys["mjj_bin_width"])

bkg = df.loc[df[LABEL_COL] == 0, RESONANT_FEATURE]
sig = df.loc[df[LABEL_COL] == 1, RESONANT_FEATURE]

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.hist(bkg, bins=bins, histtype="step", lw=1.6, label="QCD (fondo)")
ax.hist(sig, bins=bins, histtype="stepfilled", alpha=0.35,
        label=r"$W' \to XY$ (segnale, tutti i 100k)")

for edge in phys["signal_region"]:
    ax.axvline(edge, ls="--", color="crimson", lw=1.2)
for lo, hi in phys["sidebands"]:
    ax.axvspan(lo, hi, color="grey", alpha=0.12)

ax.set_yscale("log")
ax.set_xlabel(r"$m_{JJ}$ [GeV]")
ax.set_ylabel(f"Eventi / {phys['mjj_bin_width']:.0f} GeV")
ax.set_title("Rosso = signal region, grigio = sidebands")
ax.legend()
fig.tight_layout()
fig.savefig(FIGDIR / "00_mjj_distribution.pdf")

## 3. Le variabili ausiliarie $Y$

Qui si vede la fisica del segnale: due picchi netti nelle masse dei jet a 100 e 500 GeV
(da cui $\Delta m \simeq 400$ GeV) e $\tau_{21}$ spostata verso il basso, perche' i jet
di segnale sono a due prong mentre quelli QCD sono a un prong.

In [ ]:
ranges = {"mja": (0, 800), "delta_m": (0, 800), "tau21a": (0, 1.2), "tau21b": (0, 1.2)}

fig, axes = plt.subplots(2, 2, figsize=(10, 7))
for ax, col in zip(axes.ravel(), AUX_FEATURES):
    b = np.linspace(*ranges[col], 60)
    ax.hist(df.loc[df[LABEL_COL] == 0, col], bins=b, density=True,
            histtype="step", lw=1.6, label="fondo")
    ax.hist(df.loc[df[LABEL_COL] == 1, col], bins=b, density=True,
            histtype="stepfilled", alpha=0.35, label="segnale")
    ax.set_xlabel(col)
    ax.set_ylabel("densita\u0300")
axes[0, 0].legend()
fig.tight_layout()
fig.savefig(FIGDIR / "00_aux_features.pdf")

## 4. Correlazione con $m_{JJ}$ (controllo critico)

Va misurata **sul solo fondo**: e' il fondo che non deve essere scolpito dal taglio.
Correlazioni con $|\rho| \gtrsim 0.2$ sono un campanello d'allarme e richiedono
decorrelazione (adversarial, DDT, o riscalatura delle masse con $m_{JJ}$).

In [ ]:
corr = correlation_with_mjj(df[df[LABEL_COL] == 0])
print(corr.round(4).to_string())

fig, ax = plt.subplots(figsize=(6, 3.2))
colors = ["crimson" if abs(v) > 0.2 else "steelblue" for v in corr.values]
ax.barh(corr.index, corr.values, color=colors)
ax.axvline(0, color="k", lw=0.8)
for x in (-0.2, 0.2):
    ax.axvline(x, ls=":", color="crimson", lw=1)
ax.set_xlabel(r"Pearson $\rho$ con $m_{JJ}$ (solo fondo)")
fig.tight_layout()
fig.savefig(FIGDIR / "00_correlations.pdf")

### Controllo piu' fine: profilo di $\langle Y \rangle$ in bin di $m_{JJ}$

La correlazione di Pearson vede solo la dipendenza lineare. Un profilo per bin
rivela anche andamenti non monotoni, che sono comunque pericolosi.

In [ ]:
b = df[df[LABEL_COL] == 0]
edges = np.linspace(*phys["mjj_range"], 19)
centers = 0.5 * (edges[:-1] + edges[1:])
idx = np.digitize(b[RESONANT_FEATURE], edges) - 1

fig, axes = plt.subplots(1, 4, figsize=(14, 3.2), sharex=True)
for ax, col in zip(axes, AUX_FEATURES):
    vals = b[col].to_numpy()
    mean = np.array([vals[idx == i].mean() if (idx == i).sum() > 20 else np.nan
                     for i in range(len(centers))])
    err = np.array([vals[idx == i].std() / max(np.sqrt((idx == i).sum()), 1)
                    if (idx == i).sum() > 20 else np.nan
                    for i in range(len(centers))])
    ax.errorbar(centers, mean, yerr=err, fmt="o-", ms=3, lw=1)
    ax.set_xlabel(r"$m_{JJ}$ [GeV]")
    ax.set_title(rf"$\langle$ {col} $\rangle$", fontsize=10)
fig.tight_layout()
fig.savefig(FIGDIR / "00_profile_vs_mjj.pdf")

## 5. Quanti eventi ci sono nelle finestre?

Serve per dimensionare l'iniezione: la statistica in SR determina quanto segnale
il classificatore riesce a vedere.

In [ ]:
m = df[RESONANT_FEATURE]
lo, hi = phys["signal_region"]
in_sr = (m >= lo) & (m < hi)
in_sb = np.zeros(len(df), dtype=bool)
for a, bb in phys["sidebands"]:
    in_sb |= ((m >= a) & (m < bb)).to_numpy()

for name, mask in [("signal region", in_sr.to_numpy()), ("sidebands", in_sb)]:
    n_b = int(((df[LABEL_COL] == 0) & mask).sum())
    n_s = int(((df[LABEL_COL] == 1) & mask).sum())
    print(f"{name:<15} fondo={n_b:>8,}  segnale disponibile={n_s:>7,}")

n_inj = phys["n_signal_injected"]
n_b_sr = int(((df[LABEL_COL] == 0) & in_sr.to_numpy()).sum())
frac_in_sr = ((df[LABEL_COL] == 1) & in_sr.to_numpy()).sum() / max((df[LABEL_COL] == 1).sum(), 1)
s_sr = n_inj * frac_in_sr
print(f"\nIniettando {n_inj} eventi di segnale:")
print(f"  S in SR  = {s_sr:.0f}")
print(f"  S/B      = {s_sr / n_b_sr:.2e}")
print(f"  S/sqrt(B)= {s_sr / np.sqrt(n_b_sr):.2f}")

L'ultimo numero e' il punto di partenza: nell'articolo di Collins, Howe e Nachman
si parte da $S/\sqrt{B} \simeq 1.8$, cioe' un eccesso del tutto trascurabile,
e si arriva a $7\sigma$ dopo il taglio del classificatore.

**Prossimo notebook:** `01_benchmark_supervised.ipynb`.